# Training with Frozen Genre Classifier Loss

This notebook demonstrates:
1. Creating a frozen genre classifier loss class
2. Integrating it with DiT and FlowMatching models
3. Using it during training to guide style transfer
4. Monitoring gradient flow and loss convergence

## Section 1: Import Required Libraries and Dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from typing import Dict, Tuple, List

# Import from the project
from models.dit import DiT
from models.flow import FlowMatching
from models.genre_classifier_loss import GenreClassifierLoss, GenreClassifierAuxiliaryLoss
from training.training import TrainingPipeline, TrainingConfig
from training.dataloader import RandomPairMelDataset

print("✓ All dependencies imported successfully")
print(f"  PyTorch version: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## Section 2: Inspect the Frozen Genre Classifier Loss Class

The `GenreClassifierLoss` class has already been created in `models/genre_classifier_loss.py`. Let's examine its key features:

In [ ]:
# Instantiate the genre classifier loss
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

genre_loss = GenreClassifierLoss(
    num_genres=2,
    mel_height=100,
    mel_width=512,
    embedding_dim=128,
    loss_weight=0.5,
    device=device
)

print("✓ Genre Classifier Loss Module Created")
print(f"  Architecture:")
print(genre_loss.classifier)
print(f"\n  Number of parameters: {sum(p.numel() for p in genre_loss.classifier.parameters()):,}")
print(f"  All parameters frozen: {not any(p.requires_grad for p in genre_loss.classifier.parameters())}")
print(f"  Loss weight: {genre_loss.loss_weight}")

## Section 3: Create Models (DiT + FlowMatching with Integrated Genre Loss)

In [ ]:
# Create DiT model
dit = DiT(
    in_channels=1,
    patch_height=10,
    patch_width=4,
    embed_dim=256,
    num_blocks=4,
    num_heads=4,
    hidden_dim=1024,
    num_genres=2,
    dropout=0.1,
).to(device)

print(f"✓ DiT Model Created")
print(f"  Parameters: {sum(p.numel() for p in dit.parameters()):,}")

# Create FlowMatching with integrated genre classifier loss
flow = FlowMatching(
    dit,
    use_genre_loss=True,      # Enable genre classifier loss
    genre_loss_weight=0.5,     # Weight of genre loss in total loss
    num_genres=2,
    mel_height=100,
    mel_width=512,
).to(device)

print(f"\n✓ FlowMatching Model Created with Frozen Genre Classifier Loss")
print(f"  Total parameters: {sum(p.numel() for p in flow.parameters()):,}")
print(f"  Genre classifier parameters: {sum(p.numel() for p in flow.genre_classifier_loss.classifier_loss.classifier.parameters()):,}")
print(f"  Genre loss enabled: {flow.use_genre_loss}")
print(f"  Genre loss weight: {flow.genre_loss_weight}")

## Section 4: Test Loss Computation on Sample Data

In [ ]:
# Create dummy batch of mel spectrograms
batch_size = 2
mel_height = 100
mel_width = 512

# x0: source mels (classical)
x0 = torch.randn(batch_size, 1, mel_height, mel_width).to(device)

# x1: target mels (rock)
x1 = torch.randn(batch_size, 1, mel_height, mel_width).to(device)

# genre_ids: target genres (1 for rock)
genre_ids = torch.ones(batch_size, dtype=torch.long, device=device)

print(f"Sample data shapes:")
print(f"  x0 (source): {x0.shape}")
print(f"  x1 (target): {x1.shape}")
print(f"  genre_ids: {genre_ids.shape}")

# Set models to training mode
flow.train()

# Compute loss
print(f"\nComputing combined loss (Flow Matching + Genre Classifier)...")
loss, loss_dict = flow.compute_loss(x0, x1, genre_ids)

print(f"\n✓ Loss computation successful!")
print(f"  Total loss: {loss.item():.6f}")
print(f"  Loss components:")
for key, value in loss_dict.items():
    print(f"    {key}: {value:.6f}")

## Section 5: Verify Gradient Flow (Frozen Classifier)

In [ ]:
# Perform backward pass to compute gradients
print("Performing backward pass...")
flow.zero_grad()
loss.backward()
print("✓ Backward pass completed!")

# Check gradient flow
print("\nGradient Flow Analysis:")
print("-" * 60)

# Check DiT gradients (should have non-zero gradients)
dit_grads = [p.grad for p in dit.parameters() if p.grad is not None]
print(f"✓ DiT parameters with gradients: {len(dit_grads)}/{sum(1 for p in dit.parameters())}")

if dit_grads:
    grad_magnitudes = [g.abs().mean().item() for g in dit_grads]
    print(f"  Mean gradient magnitude: {np.mean(grad_magnitudes):.6f}")
    print(f"  Max gradient magnitude: {np.max(grad_magnitudes):.6f}")
    print(f"  Min gradient magnitude: {np.min(grad_magnitudes):.6f}")

# Check genre classifier gradients (should be zero - frozen)
classifier_params = flow.genre_classifier_loss.classifier_loss.classifier.parameters()
classifier_grads = [p.grad for p in classifier_params if p.grad is not None]

print(f"\n✓ Genre Classifier parameters with gradients: {len(classifier_grads)} (should be 0 - FROZEN)")
if len(classifier_grads) == 0:
    print("  ✓ Confirmed: Genre classifier is FROZEN (no gradients)")
else:
    print("  ⚠ WARNING: Genre classifier has gradients (should be frozen!)")

print(f"\n✓ Loss computation verifies:")
print(f"  1. Flow matching loss guides DiT training")
print(f"  2. Genre classifier is frozen (no gradients)")
print(f"  3. Combined loss flows through trainable models only")

## Section 6: Visualize Gradient Distribution Across Layers

In [ ]:
# Collect gradient statistics for visualization
layer_names = []
layer_grad_means = []
layer_grad_stds = []

print("Analyzing gradient distribution across DiT layers...\n")

for name, param in dit.named_parameters():
    if param.grad is not None:
        grad_mean = param.grad.abs().mean().item()
        grad_std = param.grad.abs().std().item()
        layer_grad_means.append(grad_mean)
        layer_grad_stds.append(grad_std)
        
        # Simplify layer names for readability
        simple_name = name.split('.')[-2] + '.' + name.split('.')[-1]
        layer_names.append(simple_name)

# Plot gradient magnitudes
if layer_grad_means:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Plot 1: Gradient means with error bars
    x_pos = np.arange(len(layer_names))
    axes[0].bar(x_pos, layer_grad_means, yerr=layer_grad_stds, capsize=5, alpha=0.7, color='steelblue')
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel('Mean Gradient Magnitude')
    axes[0].set_title('Gradient Flow Through DiT Layers (with std deviation)')
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(layer_names, rotation=45, ha='right')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Plot 2: Log scale for better visualization
    axes[1].semilogy(x_pos, layer_grad_means, marker='o', linewidth=2, markersize=8, color='coral')
    axes[1].fill_between(x_pos, 
                          np.array(layer_grad_means) - np.array(layer_grad_stds),
                          np.array(layer_grad_means) + np.array(layer_grad_stds),
                          alpha=0.3, color='coral')
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('Gradient Magnitude (log scale)')
    axes[1].set_title('Gradient Flow Through DiT (Log Scale)')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(layer_names, rotation=45, ha='right')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Gradient visualization complete")
    print(f"  Total layers analyzed: {len(layer_names)}")
    print(f"  Layers with gradients: {len(layer_grad_means)}")
else:
    print("No gradients to visualize")

## Section 7: Integrate with Training Pipeline

In [ ]:
# Create training configuration
config = TrainingConfig(
    num_epochs=3,              # Demo with 3 epochs
    batch_size=2,
    learning_rate=1e-4,
    weight_decay=1e-4,
    grad_clip_norm=1.0,
    checkpoint_interval=1,
    checkpoint_dir="checkpoints/demo_with_genre_loss",
)

# Initialize training pipeline
# This automatically uses FlowMatching with genre loss enabled
pipeline = TrainingPipeline(config)

print(f"✓ Training Pipeline Created")
print(f"  Configuration:")
print(f"    Epochs: {config.num_epochs}")
print(f"    Batch size: {config.batch_size}")
print(f"    Learning rate: {config.learning_rate}")
print(f"    Loss weight (genre): {flow.genre_loss_weight}")
print(f"\n  Models:")
print(f"    DiT parameters: {sum(p.numel() for p in pipeline.dit.parameters()):,}")
print(f"    Flow parameters: {sum(p.numel() for p in pipeline.flow.parameters()):,}")
print(f"    Genre classifier frozen: ✓")

# Show the training loop would work
print(f"\n✓ Training pipeline is ready")
print(f"  To train, call: pipeline.train(dataloader)")
print(f"  Loss components will be logged during training:")

In [ ]:
print("  - Total Loss:       Flow Matching + Genre Classifier Loss")
print("  - Flow Loss:        Base transformation loss (MSE)")
print("  - Genre Consistency: Contrastive loss between features")
print("  - Genre Alignment:   MSE to genre prototypes")
print("\nExample training output format:")
print("  Epoch 1/3 - Step 10/50 - Loss: 0.4523 | Flow: 0.3890 | Genre: 0.0633")

## Section 8: Simulate Training Loop with Loss Monitoring

In [ ]:
# Simulate a mini training loop with loss tracking
num_steps = 10
training_losses = []
flow_losses = []
genre_losses = []

optimizer = torch.optim.AdamW(
    flow.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

flow.train()

print("Simulating 10 training steps...\n")
print(f"{'Step':<6} {'Total Loss':<15} {'Flow Loss':<15} {'Genre Loss':<15}")
print("-" * 60)

for step in range(1, num_steps + 1):
    # Create random batch
    x0 = torch.randn(batch_size, 1, mel_height, mel_width).to(device)
    x1 = torch.randn(batch_size, 1, mel_height, mel_width).to(device)
    genre_ids = torch.ones(batch_size, dtype=torch.long, device=device)
    
    # Forward pass
    loss, loss_dict = flow.compute_loss(x0, x1, genre_ids)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(flow.parameters(), config.grad_clip_norm)
    optimizer.step()
    
    # Track losses
    training_losses.append(loss.item())
    flow_losses.append(loss_dict.get('flow_matching_loss', 0))
    genre_losses.append(loss_dict.get('genre_classifier_loss', 0))
    
    # Print progress
    print(f"{step:<6} {loss.item():<15.6f} {loss_dict.get('flow_matching_loss', 0):<15.6f} {loss_dict.get('genre_classifier_loss', 0):<15.6f}")

print("\n✓ Training simulation complete!")
print(f"  Final total loss: {training_losses[-1]:.6f}")
print(f"  Loss trend: {'↓ Decreasing' if training_losses[-1] < training_losses[0] else '↑ Increasing'}")

## Section 9: Visualize Loss Convergence

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: All losses together
axes[0, 0].plot(training_losses, marker='o', label='Total Loss', linewidth=2, markersize=8, color='navy')
axes[0, 0].plot(flow_losses, marker='s', label='Flow Matching Loss', linewidth=2, markersize=6, color='green')
axes[0, 0].plot(genre_losses, marker='^', label='Genre Classifier Loss', linewidth=2, markersize=6, color='red')
axes[0, 0].set_xlabel('Training Step')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Combined Loss Components During Training')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Loss ratio (genre loss / total loss)
loss_ratios = [g / (t + 1e-6) for g, t in zip(genre_losses, training_losses)]
axes[0, 1].plot(loss_ratios, marker='o', color='purple', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Training Step')
axes[0, 1].set_ylabel('Genre Loss / Total Loss')
axes[0, 1].set_title('Genre Loss Contribution Over Time')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Target ratio (50%)')
axes[0, 1].legend()

# Plot 3: Flow loss trend
axes[1, 0].plot(flow_losses, marker='o', color='green', linewidth=2, markersize=8)
axes[1, 0].fill_between(range(len(flow_losses)), flow_losses, alpha=0.3, color='green')
axes[1, 0].set_xlabel('Training Step')
axes[1, 0].set_ylabel('Flow Matching Loss')
axes[1, 0].set_title('Flow Matching Loss Trajectory')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Genre loss trend
axes[1, 1].plot(genre_losses, marker='o', color='red', linewidth=2, markersize=8)
axes[1, 1].fill_between(range(len(genre_losses)), genre_losses, alpha=0.3, color='red')
axes[1, 1].set_xlabel('Training Step')
axes[1, 1].set_ylabel('Genre Classifier Loss')
axes[1, 1].set_title('Genre Classifier Loss Trajectory')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Loss visualization complete")
print("\nInterpretation:")
print(f"  - Total loss started at {training_losses[0]:.6f}")
print(f"  - Total loss ended at {training_losses[-1]:.6f}")
print(f"  - Change: {training_losses[-1] - training_losses[0]:.6f}")
print(f"  - Genre loss contribution: {np.mean([g/t for g, t in zip(genre_losses, training_losses)]):.1%}")

## Section 10: Summary and Key Takeaways

In [ ]:
print("""
═══════════════════════════════════════════════════════════════════════════════
                    FROZEN GENRE CLASSIFIER LOSS INTEGRATION
═══════════════════════════════════════════════════════════════════════════════

✓ SUMMARY OF WHAT WAS ACCOMPLISHED:

1. ✓ Created GenreClassifierLoss as a separate class in models/genre_classifier_loss.py
   - Frozen CNN-based genre classifier
   - Extracts 128-dimensional feature embeddings
   - Two loss components: Consistency + Alignment

2. ✓ Integrated with FlowMatching model (models/flow.py)
   - FlowMatching now accepts use_genre_loss parameter
   - compute_loss() returns loss dictionary with components
   - Proper device management for genre classifier

3. ✓ Updated TrainingPipeline (training/training.py)
   - Automatically initializes FlowMatching with genre loss
   - Training loop logs loss components separately
   - No breaking changes to existing code

4. ✓ Verified gradient flow
   - DiT/FlowMatching parameters receive gradients ✓
   - Genre classifier parameters are FROZEN (no gradients) ✓
   - Loss flows only through trainable models ✓

═══════════════════════════════════════════════════════════════════════════════

🎯 KEY FEATURES:

• Frozen Classifier: Pre-trained weights don't change during training
  └─ Prevents catastrophic forgetting of genre features
  └─ Reduces computational overhead
  └─ Provides stable feature extraction

• Dual Loss Approach:
  1. Flow Matching Loss: Guides mel spectrogram transformation
  2. Genre Loss: Ensures output matches target genre characteristics

• Loss Components:
  ├─ Genre Consistency: Contrastive loss (info-NCE style)
  ├─ Genre Alignment: MSE to genre prototypes
  └─ Combined Weight: 0.5 (adjustable)

═══════════════════════════════════════════════════════════════════════════════

📊 TRAINING MONITORING:

During training, you'll see:
  Epoch 1/10 - Step 10/50 - Loss: 0.4523 | Flow: 0.3890 | Genre: 0.0633
                                          ↓              ↓
                                   Core transformation  Genre guidance

Good training signs:
  • Flow loss decreases ✓
  • Genre loss decreases ✓
  • Gradients present in DiT ✓
  • Genre classifier frozen ✓

═══════════════════════════════════════════════════════════════════════════════

🚀 NEXT STEPS:

1. Prepare your data (source mel + target mel pairs)
2. Create training dataloader
3. Run: pipeline.train(dataloader)
4. Monitor loss components for convergence
5. Save checkpoints with genre loss metrics

Example usage:
  config = TrainingConfig(num_epochs=50, batch_size=8)
  pipeline = TrainingPipeline(config)
  loader = pipeline.setup_data(source_files, target_files)
  losses = pipeline.train(loader)

═══════════════════════════════════════════════════════════════════════════════

📚 DOCUMENTATION:

See GENRE_CLASSIFIER_LOSS_GUIDE.md for:
  • Detailed hyperparameter tuning
  • Troubleshooting common issues
  • Advanced usage patterns
  • Future enhancement ideas

═══════════════════════════════════════════════════════════════════════════════
""")